# 🏆 [06] Scoring — Crypto ML Project

**Objective:** Formally evaluate all supervised and unsupervised models with standard metrics, run a simple backtesting simulation, and generate the final consolidated report.

**Dependencies:**
- `feature/supervised` → `data/processed/features.parquet`, `models/*.pkl`
- `feature/unsupervised` → `data/features_clustering.csv`, `data/cluster_labels.csv`

**Outputs:**
- `reports/metrics_summary.csv`
- `reports/model_report.pdf`
- `reports/figures/17_*` through `reports/figures/23_*`

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from datetime import datetime

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from xgboost import XGBClassifier

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

print('All imports loaded successfully.')

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────
ROOT         = Path('.').resolve().parent
DATA_PATH    = ROOT / 'data' / 'processed' / 'features.parquet'
CLUST_FEAT   = ROOT / 'data' / 'features_clustering.csv'
CLUST_LABELS = ROOT / 'data' / 'cluster_labels.csv'
MODELS_DIR   = ROOT / 'models'
REPORTS_DIR  = ROOT / 'reports'
FIGURES_DIR  = REPORTS_DIR / 'figures'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Feature columns — same as supervised.py
FEATURE_COLS = [
    'sma_7', 'sma_14', 'sma_30',
    'ema_14', 'rsi_14',
    'bb_high', 'bb_low', 'bb_width',
    'macd', 'macd_signal',
    'Return',
]
TARGET_CLF = 'target'
TARGET_REG = 'Close'
RANDOM_STATE = 42

# Clustering parameters — same as clustering.py
K_BEST = 2
K_ALT  = 4

print(f'Root: {ROOT}')
print(f'Features parquet exists: {DATA_PATH.exists()}')
print(f'Clustering features exist: {CLUST_FEAT.exists()}')

## 1. Data Loading & Temporal Split

In [ ]:
# Load features parquet
df = pd.read_parquet(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.dropna(subset=FEATURE_COLS + [TARGET_CLF, TARGET_REG])

print(f'Shape: {df.shape}')
print(f'Coins: {sorted(df["Symbol"].unique())}')
print(f'Date range: {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'\nTarget balance:')
df[TARGET_CLF].value_counts()

In [ ]:
# Temporal split (80/10/10) — no shuffle, respects time order
dates = df['Date'].sort_values().unique()
n = len(dates)
cut_val  = dates[int(n * 0.80)]
cut_test = dates[int(n * 0.90)]

train = df[df['Date'] <  cut_val].copy()
val   = df[(df['Date'] >= cut_val) & (df['Date'] < cut_test)].copy()
test  = df[df['Date'] >= cut_test].copy()

print(f'Train : {len(train):>6} rows  ({train["Date"].min().date()} → {train["Date"].max().date()})')
print(f'Val   : {len(val):>6} rows  ({val["Date"].min().date()} → {val["Date"].max().date()})')
print(f'Test  : {len(test):>6} rows  ({test["Date"].min().date()} → {test["Date"].max().date()})')

In [ ]:
# Extract feature arrays for modelling
X_train, y_train_clf = train[FEATURE_COLS].values, train[TARGET_CLF].values
X_val,   y_val_clf   = val[FEATURE_COLS].values,   val[TARGET_CLF].values
X_test,  y_test_clf  = test[FEATURE_COLS].values,  test[TARGET_CLF].values

y_train_reg = train[TARGET_REG].values
y_val_reg   = val[TARGET_REG].values
y_test_reg  = test[TARGET_REG].values

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')

## 2. Classification Scoring

Train three classifiers (LogisticRegression, RandomForestClassifier, XGBClassifier) and compute:
- **Accuracy, Precision, Recall, F1, ROC-AUC**
- **Confusion Matrix** for each

In [ ]:
# Build classifier pipelines
classifiers = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced',
        )),
    ]),
    'RandomForestClassifier': Pipeline([
        ('clf', RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=5,
            random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced',
        )),
    ]),
    'XGBClassifier': Pipeline([
        ('clf', XGBClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0,
        )),
    ]),
}

print(f'Classifiers defined: {list(classifiers.keys())}')

In [ ]:
# Train and evaluate each classifier
tscv = TimeSeriesSplit(n_splits=5)
clf_results = []

for name, pipe in classifiers.items():
    print(f'Training {name}...')
    
    # Cross-validation
    cv_scores = cross_val_score(pipe, X_train, y_train_clf,
                                cv=tscv, scoring='roc_auc', n_jobs=-1)
    pipe.fit(X_train, y_train_clf)
    
    # Predictions
    y_va_pred = pipe.predict(X_val)
    y_te_pred = pipe.predict(X_test)
    
    # Probabilities for AUC
    try:
        y_va_proba = pipe.predict_proba(X_val)[:, 1]
        y_te_proba = pipe.predict_proba(X_test)[:, 1]
        auc_va = roc_auc_score(y_val_clf, y_va_proba)
        auc_te = roc_auc_score(y_test_clf, y_te_proba)
    except AttributeError:
        auc_va = auc_te = float('nan')
    
    clf_results.append({
        'Model':          name,
        'CV_AUC_mean':    round(cv_scores.mean(), 4),
        'CV_AUC_std':     round(cv_scores.std(), 4),
        'Val_Accuracy':   round(accuracy_score(y_val_clf, y_va_pred), 4),
        'Val_Precision':  round(precision_score(y_val_clf, y_va_pred), 4),
        'Val_Recall':     round(recall_score(y_val_clf, y_va_pred), 4),
        'Val_F1':         round(f1_score(y_val_clf, y_va_pred), 4),
        'Val_ROC_AUC':    round(auc_va, 4),
        'Test_Accuracy':  round(accuracy_score(y_test_clf, y_te_pred), 4),
        'Test_Precision': round(precision_score(y_test_clf, y_te_pred), 4),
        'Test_Recall':    round(recall_score(y_test_clf, y_te_pred), 4),
        'Test_F1':        round(f1_score(y_test_clf, y_te_pred), 4),
        'Test_ROC_AUC':   round(auc_te, 4),
    })

clf_df = pd.DataFrame(clf_results).set_index('Model')
best_clf_name = clf_df['Val_ROC_AUC'].idxmax()
best_clf = classifiers[best_clf_name]

print(f'\nBest classifier: {best_clf_name}')
clf_df

In [ ]:
# ── Classification comparison bar chart ────────────────────────────────
metrics_to_plot = ['Test_Accuracy', 'Test_Precision', 'Test_Recall',
                   'Test_F1', 'Test_ROC_AUC']

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(clf_df))
width = 0.15
colors_list = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f']

for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, clf_df[metric], width,
           label=metric.replace('Test_', ''), color=colors_list[i])

ax.set_xticks(x + width * 2)
ax.set_xticklabels(clf_df.index, rotation=15)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Classification Models — Test Set Metrics Comparison')
ax.legend(loc='lower right')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '17_scoring_clf_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices for all classifiers ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pipe) in zip(axes, classifiers.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test_clf, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Down (0)', 'Up (1)'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '18_scoring_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC curves overlay ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
palette = ['#4e79a7', '#f28e2b', '#e15759']

for (name, pipe), color in zip(classifiers.items(), palette):
    try:
        y_proba = pipe.predict_proba(X_test)[:, 1]
        RocCurveDisplay.from_predictions(
            y_test_clf, y_proba, ax=ax, name=name, color=color
        )
    except AttributeError:
        pass

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_title('ROC Curves — All Classifiers (Test Set)')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '19_scoring_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Regression Scoring

Train a RandomForestRegressor for next-day close price prediction and compute:
- **MAE, RMSE, MAPE, R²**

In [ ]:
# Build and train the regressor
reg_pipe = Pipeline([
    ('reg', RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
])

tscv = TimeSeriesSplit(n_splits=5)
cv_r2 = cross_val_score(reg_pipe, X_train, y_train_reg, cv=tscv,
                        scoring='r2', n_jobs=-1)
reg_pipe.fit(X_train, y_train_reg)

y_val_pred_reg  = reg_pipe.predict(X_val)
y_test_pred_reg = reg_pipe.predict(X_test)

# MAPE helper — avoid division by zero
def mape(y_true, y_pred):
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

reg_metrics = {
    'CV_R2_mean': round(cv_r2.mean(), 4),
    'CV_R2_std':  round(cv_r2.std(), 4),
    'Val_MAE':    round(mean_absolute_error(y_val_reg, y_val_pred_reg), 4),
    'Val_RMSE':   round(np.sqrt(mean_squared_error(y_val_reg, y_val_pred_reg)), 4),
    'Val_MAPE':   round(mape(y_val_reg, y_val_pred_reg), 4),
    'Val_R2':     round(r2_score(y_val_reg, y_val_pred_reg), 4),
    'Test_MAE':   round(mean_absolute_error(y_test_reg, y_test_pred_reg), 4),
    'Test_RMSE':  round(np.sqrt(mean_squared_error(y_test_reg, y_test_pred_reg)), 4),
    'Test_MAPE':  round(mape(y_test_reg, y_test_pred_reg), 4),
    'Test_R2':    round(r2_score(y_test_reg, y_test_pred_reg), 4),
}

print('Regression Metrics:')
for k, v in reg_metrics.items():
    print(f'  {k:<15}: {v}')

In [ ]:
# ── Predicted vs Actual scatter plot ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test_reg, y_test_pred_reg, alpha=0.3, s=10, color='#4e79a7')

min_val = min(y_test_reg.min(), y_test_pred_reg.min())
max_val = max(y_test_reg.max(), y_test_pred_reg.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5,
        label='Perfect Prediction')

ax.set_xlabel('Actual Close Price')
ax.set_ylabel('Predicted Close Price')
ax.set_title('Regression — Predicted vs Actual (Test Set)')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '20_scoring_regression_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Residual analysis ──────────────────────────────────────────────────
residuals = y_test_reg - y_test_pred_reg

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_pred_reg, residuals, alpha=0.3, s=10, color='#e15759')
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_xlabel('Predicted Close Price')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=50, color='#76b7b2', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.suptitle('Regression Residuals Analysis', fontsize=14)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '21_scoring_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Clustering Scoring

Re-run clustering algorithms and compute:
- **Silhouette Score** (higher = better)
- **Davies-Bouldin Index** (lower = better)
- **Calinski-Harabasz Index** (higher = better)

In [ ]:
# Load clustering features
features_cl = pd.read_csv(CLUST_FEAT, index_col='Symbol')
X_cl = features_cl.values
print(f'Clustering features: {features_cl.shape}')
features_cl.head()

In [ ]:
# Run and evaluate clustering algorithms
clustering_algos = {
    f'KMeans_K{K_BEST}': KMeans(n_clusters=K_BEST, random_state=42, n_init=10),
    f'KMeans_K{K_ALT}':  KMeans(n_clusters=K_ALT, random_state=42, n_init=10),
    'DBSCAN':            DBSCAN(eps=1.5, min_samples=2),
    f'Agglomerative_K{K_BEST}': AgglomerativeClustering(n_clusters=K_BEST),
}

clust_results = []

for name, algo in clustering_algos.items():
    labels = algo.fit_predict(X_cl)
    n_clusters = len(set(labels) - {-1})
    
    if n_clusters < 2:
        clust_results.append({
            'Model': name, 'N_Clusters': n_clusters,
            'Silhouette': np.nan, 'Davies_Bouldin': np.nan,
            'Calinski_Harabasz': np.nan,
        })
        continue
    
    # For DBSCAN: mask noise points
    mask = labels != -1
    X_ev = X_cl[mask] if name == 'DBSCAN' else X_cl
    l_ev = labels[mask] if name == 'DBSCAN' else labels
    
    if len(set(l_ev)) < 2:
        clust_results.append({
            'Model': name, 'N_Clusters': n_clusters,
            'Silhouette': np.nan, 'Davies_Bouldin': np.nan,
            'Calinski_Harabasz': np.nan,
        })
        continue
    
    clust_results.append({
        'Model':             name,
        'N_Clusters':        n_clusters,
        'Silhouette':        round(silhouette_score(X_ev, l_ev), 4),
        'Davies_Bouldin':    round(davies_bouldin_score(X_ev, l_ev), 4),
        'Calinski_Harabasz': round(calinski_harabasz_score(X_ev, l_ev), 4),
    })

clust_df = pd.DataFrame(clust_results)
clust_df

In [ ]:
# ── Clustering metrics bar charts ──────────────────────────────────────
plot_df = clust_df.dropna(subset=['Silhouette']).set_index('Model')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_c = ['#59a14f', '#e15759', '#4e79a7']
metrics_c = ['Silhouette', 'Davies_Bouldin', 'Calinski_Harabasz']
titles = ['Silhouette Score (higher=better)',
          'Davies-Bouldin Index (lower=better)',
          'Calinski-Harabasz Index (higher=better)']

for ax, metric, title, c in zip(axes, metrics_c, titles, colors_c):
    plot_df[metric].plot(kind='bar', ax=ax, color=c, edgecolor='white')
    ax.set_title(title, fontsize=10)
    ax.set_ylabel(metric.replace('_', ' '))
    ax.tick_params(axis='x', rotation=25)

plt.suptitle('Clustering Quality Metrics', fontsize=14)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '22_scoring_clustering_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Consolidated Model Comparison

All models in a single table for easy comparison.

In [ ]:
# Build consolidated summary table
summary_rows = []

# Classification models
for model in clf_df.index:
    summary_rows.append({
        'Model': model,
        'Type': 'Classification',
        'Key_Metric': 'ROC-AUC',
        'Value': clf_df.loc[model, 'Test_ROC_AUC'],
    })

# Regression model
summary_rows.append({
    'Model': 'RandomForestRegressor',
    'Type': 'Regression',
    'Key_Metric': 'R²',
    'Value': reg_metrics['Test_R2'],
})

# Clustering models
for _, row in clust_df.iterrows():
    summary_rows.append({
        'Model': row['Model'],
        'Type': 'Clustering',
        'Key_Metric': 'Silhouette',
        'Value': row.get('Silhouette', np.nan),
    })

summary_df = pd.DataFrame(summary_rows)
print('Consolidated Model Comparison:')
summary_df

## 6. Backtesting Simulation

Simple strategy: **buy** when the classifier predicts price will go up (1), **sell/stay out** otherwise. Compare against passive **buy & hold**.

In [ ]:
def run_backtest(test_df, clf_pipe, symbol='BTC'):
    """Run a simple backtest for a given symbol."""
    coin = test_df[test_df['Symbol'] == symbol].copy().sort_values('Date')
    if len(coin) < 2:
        return pd.DataFrame(), {}
    
    X_coin = coin[FEATURE_COLS].values
    preds = clf_pipe.predict(X_coin)
    
    coin = coin.reset_index(drop=True)
    coin['Prediction'] = preds
    coin['Daily_Return'] = coin['Close'].pct_change()
    coin['Strategy_Return'] = coin['Daily_Return'] * coin['Prediction']
    coin['BuyHold_Cum'] = (1 + coin['Daily_Return']).cumprod().fillna(1.0)
    coin['Strategy_Cum'] = (1 + coin['Strategy_Return']).cumprod().fillna(1.0)
    
    summary = {
        'Symbol': symbol,
        'Test_Days': len(coin),
        'Days_In_Market': int(coin['Prediction'].sum()),
        'BuyHold_Return_%': round((coin['BuyHold_Cum'].iloc[-1] - 1) * 100, 2),
        'Strategy_Return_%': round((coin['Strategy_Cum'].iloc[-1] - 1) * 100, 2),
    }
    return coin, summary

# Run backtest for BTC and ETH
bt_summaries = []
for sym in ['BTC', 'ETH']:
    bt_df, bt_sum = run_backtest(test, best_clf, symbol=sym)
    if bt_sum:
        bt_summaries.append(bt_sum)
        print(f'\n{sym} Backtest:')
        for k, v in bt_sum.items():
            print(f'  {k:<25}: {v}')

pd.DataFrame(bt_summaries)

In [ ]:
# ── Backtest chart — BTC ───────────────────────────────────────────────
bt_btc, _ = run_backtest(test, best_clf, symbol='BTC')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(bt_btc['Date'], bt_btc['BuyHold_Cum'], label='Buy & Hold',
        color='#4e79a7', linewidth=2)
ax.plot(bt_btc['Date'], bt_btc['Strategy_Cum'], label='Model Strategy',
        color='#e15759', linewidth=2)
ax.fill_between(bt_btc['Date'], bt_btc['BuyHold_Cum'],
                bt_btc['Strategy_Cum'], alpha=0.15, color='gray')
ax.set_title('Backtesting — BTC (Test Period)', fontsize=14)
ax.set_ylabel('Cumulative Return (1 = starting capital)')
ax.set_xlabel('Date')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '23_scoring_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export Results

Save `metrics_summary.csv` and generate the PDF report via `src/scoring.py`.

In [ ]:
# ── Save metrics CSV ───────────────────────────────────────────────────
all_rows = []

# Classification
for model in clf_df.index:
    row = clf_df.loc[model].to_dict()
    row['Model'] = model
    row['Type'] = 'Classification'
    all_rows.append(row)

# Regression
reg_row = reg_metrics.copy()
reg_row['Model'] = 'RandomForestRegressor'
reg_row['Type'] = 'Regression'
all_rows.append(reg_row)

# Clustering
for _, r in clust_df.iterrows():
    r_dict = r.to_dict()
    r_dict['Type'] = 'Clustering'
    all_rows.append(r_dict)

# Backtesting
for bt in bt_summaries:
    bt_row = bt.copy()
    bt_row['Type'] = 'Backtesting'
    bt_row['Model'] = f"Backtest_{bt['Symbol']}"
    all_rows.append(bt_row)

metrics_out = pd.DataFrame(all_rows)
priority = ['Model', 'Type']
other_cols = [c for c in metrics_out.columns if c not in priority]
metrics_out = metrics_out[priority + other_cols]

metrics_out.to_csv(REPORTS_DIR / 'metrics_summary.csv', index=False)
print(f'Saved: {REPORTS_DIR / "metrics_summary.csv"}')
metrics_out

In [ ]:
# ── Save models ────────────────────────────────────────────────────────
joblib.dump(best_clf, MODELS_DIR / 'best_classifier.pkl')
joblib.dump(reg_pipe, MODELS_DIR / 'best_regressor.pkl')
print(f'Saved: {MODELS_DIR / "best_classifier.pkl"} ({best_clf_name})')
print(f'Saved: {MODELS_DIR / "best_regressor.pkl"}')

In [ ]:
# ── Generate PDF report (run the standalone script) ────────────────────
# The PDF is generated by src/scoring.py. You can run it from the terminal:
#   python src/scoring.py
#
# Or directly call the PDF generator:
import sys
sys.path.insert(0, str(ROOT / 'src'))

from scoring import generate_pdf_report

generate_pdf_report(
    clf_df, 
    {'Test_MAE': reg_metrics['Test_MAE'], 'Test_RMSE': reg_metrics['Test_RMSE'],
     'Test_MAPE': reg_metrics['Test_MAPE'], 'Test_R2': reg_metrics['Test_R2'],
     'CV_R2_mean': reg_metrics['CV_R2_mean']},
    clust_df,
    bt_summaries,
    ['BTC', 'ETH']
)
print('\nPDF report generated successfully!')

## 8. Key Findings & Conclusions

### Classification
- All three classifiers perform close to random (AUC ≈ 0.50), which is expected for daily cryptocurrency price direction prediction.
- **RandomForestClassifier** achieved the highest validation AUC, though marginal.
- The balanced class distribution (≈50/50) rules out class imbalance as a confounding factor.

### Regression
- The regressor achieves **high R² on validation** (0.998) thanks to the auto-correlated nature of prices.
- **Test R² drops to ~0.46**, revealing that the model struggles to generalize to the 2020-2021 price explosion.
- Large MAPE on test is driven by the extreme BTC/ETH price jump during the test period.

### Clustering
- **KMeans K=2** and **Agglomerative K=2** produce identical results (Silhouette=0.67), cleanly separating outlier coins.
- **KMeans K=4** offers more granular grouping with still-decent Silhouette (0.53).
- DBSCAN collapses to a single cluster with the chosen hyperparameters.

### Backtesting
- The model-based strategy **underperforms buy & hold** during the 2020-2021 bull market, which is expected given near-random classification accuracy.
- This confirms that the models lack sufficient signal for profitable trading strategies in isolation.